# 02 — Rapport GHU, étape par étape

Ce notebook **reproduit chaque section** de la page produite par
`report_builder.build_rapport_ghu`, avec les **graphiques inline**, pour le GHU
`GHU Nord` (exemple du notebook). La vue page complète (IFrame) est conservée à la fin.

> ⚠ Duplique volontairement la logique de `build_rapport_ghu`. Re-synchroniser si
> `report_builder` évolue.

In [1]:
# Parametre papermill - modifiable
GHU_NAME = 'GHU Nord'

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from pathlib import Path
import pandas as pd
from IPython.display import HTML

DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

from report_builder import (
    load_aphp, load_survival,
    appareil_counts_table, survival_delay_table,
)
from chart_utils import (
    line_evolution, donut_market_share, TREATMENT_COLS, GHU_LIST,
)

assert GHU_NAME in GHU_LIST, f'{GHU_NAME} non reconnu. Options : {GHU_LIST}'

aphp = load_aphp(DATA_DIR)
surv = load_survival(DATA_DIR)

# Sous-ensembles (identiques a build_rapport_ghu)
ghu_total  = aphp[(aphp.entite == GHU_NAME) & (aphp.appareil == 'TOTAL')]
aphp_total = aphp[(aphp.entite == 'AP-HP') & (aphp.appareil == 'TOTAL')]

years = sorted(ghu_total.annee.unique())
last_year, prev_year = years[-1], years[-2]
lv = ghu_total[ghu_total.annee == last_year].iloc[0]
pv = ghu_total[ghu_total.annee == prev_year].iloc[0]
print(f'GHU={GHU_NAME} - annees {years} - derniere={last_year}')

GHU=GHU Nord - annees [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] - derniere=2023


## Section 1 — Indicateurs clés

5 cartes KPI (dernière année vs N-1). Vue tableau équivalente :

In [2]:
mesures = [
    ('Patients (total)',       'nb_patients'),
    ('Nouveaux patients',      'nb_nouveaux_patients'),
    ('Sejours chirurgie',      'nb_sejours_chirurgie'),
    ('Sejours chimiotherapie', 'nb_sejours_chimiotherapie'),
    ('Sejours radiotherapie',  'nb_sejours_radiotherapie'),
]
pd.DataFrame([
    {'Indicateur': lab, str(prev_year): int(pv[col]), str(last_year): int(lv[col]),
     'Var. %': round((lv[col]-pv[col])/pv[col]*100, 1) if pv[col] else None}
    for lab, col in mesures
])

,Indicateur,2022,2023,Var. %
0,Patients (total),13473,13817,2.6
1,Nouveaux patients,6751,6900,2.2
2,Sejours chirurgie,5134,5244,2.1
3,Sejours chimiotherapie,5898,6020,2.1
4,Sejours radiotherapie,3862,3933,1.8


## Section 2 — Répartition entre les groupes hospitaliers (GHU)

`donut_market_share` (dernière année) + `line_evolution` des patients par GHU.
Identique au rapport AP-HP : tous les GHU sont représentés.

In [3]:
ghu_total_all = aphp[(aphp.entite.isin(GHU_LIST)) & (aphp.appareil == 'TOTAL')]
fig_repart_donut = donut_market_share(
    ghu_total_all[ghu_total_all.annee == last_year], 'entite', 'nb_patients',
    f'Repartition par GHU - {last_year}')
fig_repart_donut.show()

In [4]:
fig_repart_lines = line_evolution(ghu_total_all, 'annee', 'nb_patients', 'entite',
                                  'Patients par GHU - evolution', entities=GHU_LIST)
fig_repart_lines.show()

## Section 3 — Évolution (patients)

Part de marché du GHU dans l'AP-HP (`nb_patients(GHU) / nb_patients(AP-HP) × 100`)
+ séjours par mode de PEC du GHU.

In [5]:
# Part de marche : alignement sur les annees communes
aphp_pts = aphp_total.set_index('annee')['nb_patients']
share_data = ghu_total.copy()
share_data['part_marche'] = share_data.apply(
    lambda r: r['nb_patients'] / aphp_pts[r['annee']] * 100
    if r['annee'] in aphp_pts.index else None, axis=1)
fig_share = line_evolution(share_data, 'annee', 'part_marche', 'entite',
                           f'Part de marche dans l AP-HP - {GHU_NAME} (%)',
                           entities=[GHU_NAME])
fig_share.show()

In [6]:
melted = ghu_total.melt(id_vars=['annee'], value_vars=list(TREATMENT_COLS.keys()),
                        var_name='type_sejour', value_name='nb_sejours')
melted['label'] = melted['type_sejour'].map(TREATMENT_COLS)
fig_sejours = line_evolution(melted, 'annee', 'nb_sejours', 'label',
                             f'Sejours par mode de prise en charge - {GHU_NAME}',
                             entities=list(TREATMENT_COLS.values()))
fig_sejours.show()

## Section 4 — Analyse par appareil

Tableau chiffré `appareil_counts_table` pour ce GHU (affiché en HTML).

In [7]:
HTML(appareil_counts_table(aphp, GHU_NAME))

## Section 5 — Survie et délais de prise en charge (par appareil)

Tableau `survival_delay_table` pour ce GHU, affiché en HTML.

In [8]:
HTML(survival_delay_table(surv, aphp, GHU_NAME))

## Page complète (référence)

In [9]:
from report_builder import build_rapport_ghu

out = build_rapport_ghu(GHU_NAME, DATA_DIR, OUTPUT_DIR)
print(f'Rapport genere : {out}')

from IPython.display import IFrame
IFrame(os.path.relpath(out), width='100%', height=800)

  → ../output/rapport_ghu_nord.html
Rapport genere : ../output/rapport_ghu_nord.html
